# 3. Proportional Control

[Previous lesson](02_open_loop_control.ipynb) · [Next lesson](04_integral_control.ipynb)

Close the loop: measure speed, calculate the error, and choose voltage from that error. We will build a sampled simulation without confusing controller updates with numerical solver steps.

Work through the explanations, then implement each exercise. Hints and complete reference solutions are collapsed below your code cells. Solutions build in order; Run All does not execute them.

**Prerequisites:** complete the preceding lessons. This notebook starts independently: run the setup cell once before attempting the exercises.

## Previously implemented code

This setup repeats earlier material so you do not need another notebook running. Read it as a recap, then run it. The new work for this lesson remains in the empty cells below.

**Setup inventory:** `motor_dynamics` evaluates the two motor derivatives using SI units and state order `[current, omega]`. Where supplied, `simulate` advances the sampled controller and motor separately, and `plot_run` displays speed, voltage, and term contributions. Controller classes in setup were taught previously; their histories start fresh when you construct an instance. Run setup after restarting the kernel, then execute your exercise cells from top to bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

R, L, K_t, K_e, J, b = 1.0, 0.01, 0.1, 0.1, 0.01, 0.001

def motor_dynamics(t, state, voltage, load_torque):
    current, omega = state
    return [(voltage - R*current - K_e*omega)/L,
            (K_t*current - b*omega - load_torque)/J]


<img src="assets/lesson-03.svg" alt="Negative feedback with sampled proportional control" style="display:block; width:100%; height:auto; margin:0 auto;">

<div class="figure-caption" style="background-color:rgba(128,128,128,0.10);padding:12px 18px;">

**Negative feedback.** Subtract measured speed from target. The controller updates once per sample; its voltage is held while the motor evolves.

</div>

## 3.1 Negative feedback and proportional gain

Define speed error as target minus measurement:

$$e_k=r_k-\omega_k,\qquad V_k=K_p e_k.$$

Positive error requests positive voltage. If the speed rises above target, error becomes negative and the ideal bidirectional drive requests braking voltage. The units of $K_p$ are V/(rad/s). At a 40 rad/s error and $K_p=0.4$, the command is 16 V. Unlimited voltage is an intentional simplification until Lesson 9.

### Trace one feedback decision

Suppose the target is 40 rad/s and the measured speed is 30 rad/s. The error is 10 rad/s. At $K_p=0.4$ V/(rad/s), the command is 4 V. If speed instead measures 42 rad/s, the error is −2 and the command is −0.8 V. The controller reacts to the sign of the mismatch, not just its magnitude.

The chosen subtraction implements negative feedback: increasing measured speed decreases commanded voltage. Reversing the subtraction while keeping positive gain would reinforce deviations instead of opposing them.

This control law has no memory. Two updates with the same target and measured speed give the same command, regardless of earlier errors. The `dt` argument is accepted for a consistent controller interface, but P does not use it.

At startup, a 40 rad/s error requests 16 V. We are intentionally using an ideal voltage source. Later we will limit voltage to ±6 V; that change will alter the response, even with the same gain. Always inspect command effort alongside speed.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 1 — Implement proportional control

Implement PController with update(setpoint, measurement, dt). Expose p_term, i_term, d_term for logging, with I and D zero. Check a 40 rad/s error at gain 0.4 gives 16 V.

</div>

<details>
<summary>Show a hint</summary>

The time step is part of a shared interface but is not needed for the proportional calculation.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
class PController:
    def __init__(self, kp):
        self.kp = kp
        self.i_term = self.d_term = 0.0
    def update(self, setpoint, measurement, dt):
        self.p_term = self.kp*(setpoint-measurement)
        return self.p_term

c = PController(.4)
assert c.update(40,0,.001)==16.0
assert c.update(40,50,.001)==-4.0
```

The sign is essential: subtracting target from measurement would turn the feedback in the wrong direction.

</details>

## 3.2 Separate sampling from integration

A digital controller observes the system at discrete times. At each sample, calculate one command, hold it constant for the next interval, and integrate the motor. This is a zero-order hold.

We use $\Delta t=0.001$ s. That is smaller than the winding timescale of 0.01 s, but small time steps alone do not establish closed-loop stability. We will check representative results with a finer interval.

<div class="lesson-callout" style="border-left: 3px solid #b99a62; background-color: rgba(185, 154, 98, 0.08); padding: 12px 18px; margin: 16px 0;">

**Implementation note**

Never update controller history inside the ODE callback. Adaptive solvers may evaluate that callback multiple times and revisit times. Controller updates belong in the outer fixed-rate loop.

</div>

### One sample interval, in order

At time $t_k$, the state contains current and speed before the next command has acted. Read that speed, compute error, call the controller exactly once, and hold the returned voltage during $[t_k,t_{k+1})$. Only then integrate the motor to obtain the next state.

The integrator may take several internal steps during that 1 ms interval. Those steps improve its estimate of the continuous motor response; they are not extra sensor samples or controller updates. The motor callback is therefore a pure derivative calculation with a fixed voltage argument during each interval.

Log a row before advancing the state. In that row, the speed is the sample that caused the command, and the voltage is the command that will act next. This timing convention avoids silently pairing a new command with a future measurement. With a duration of 8 s, the final logged time is 7.999 s; there are 8,000 intervals and rows.

Use named log fields so later plots do not depend on remembering numerical column positions. Store true speed separately from measured speed, even while both are identical. The optional noise argument introduced in the full helper will let later lessons perturb only the measurement.

### Python tools for the sample loop

A **callback** is a function you pass to the simulator so it can ask for a value at time `t`. `lambda t: 40.0` is shorthand for a function that always returns 40. A step schedule, `lambda t: 20.0 if t < 2 else 40.0`, returns different values on either side of the event.

A dictionary maps names to values: `run['speed']` retrieves the speed array. Build rows in a list using `append`, then use `np.asarray(records)` to form a two-dimensional array. `.T` transposes rows into columns; `dict(zip(keys, columns))` pairs each column with its field name. `getattr(controller, 'requested', voltage)` reads an optional attribute, falling back to the returned voltage for controllers without saturation.

Validate `dt` and duration as finite positive numbers before dividing. Require at least one whole interval, reject fractional interval counts, and reject negative/nonfinite noise standard deviations. These checks keep an invalid experiment from returning a misleading empty log.

<div class="implementation-task" style="border-left:3px solid #b77979; background-color:rgba(183,121,121,0.08); padding:12px 18px; margin:16px 0;">

### Your implementation 2 — Advance one held-voltage interval

Start at `[0,0]`. With P gain 0.4 and target 40, calculate the command once and integrate only from 0 to 0.001 s. Check command=16 V, positive final current, and positive final speed.

</div>

<details>
<summary>Show a hint</summary>

Call the controller outside solve_ivp; pass its returned voltage as a fixed argument.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
controller=PController(.4)
state=np.array([0.0,0.0])
dt=.001
command=controller.update(40.0,state[1],dt)
step=solve_ivp(motor_dynamics,(0,dt),state,args=(command,0.0),rtol=1e-7,atol=1e-9)
assert step.success and command==16.0
assert step.y[0,-1]>0 and step.y[1,-1]>0
print('State after one interval:',step.y[:,-1])
```

The command is based on the initial state. The final state will be used to calculate the next command, not to retroactively change the first one.

</details>

<div class="implementation-task" style="border-left:3px solid #b77979; background-color:rgba(183,121,121,0.08); padding:12px 18px; margin:16px 0;">

### Your implementation 3 — Repeat the sample-and-hold cycle

Repeat the same sequence for 1,000 intervals. Log time, current, speed, and voltage before each integration. Check a 1000×4 array and final logged time 0.999 s.

</div>

<details>
<summary>Show a hint</summary>

Use k*dt for time rather than repeatedly adding dt. Replace state only after a successful solve.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
state=np.zeros(2)
controller=PController(.4)
rows=[]
for k in range(1000):
    t=k*dt
    command=controller.update(40.0,state[1],dt)
    rows.append([t,*state,command])
    step=solve_ivp(motor_dynamics,(t,t+dt),state,args=(command,0.0),rtol=1e-7,atol=1e-9)
    if not step.success:raise RuntimeError(step.message)
    state=step.y[:,-1]
rows=np.array(rows)
assert rows.shape==(1000,4)
np.testing.assert_allclose(rows[-1,0],.999)
```

The loop follows exactly the same operation order as the single-interval experiment. Now generalize it without changing that order.

</details>

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 4 — Build the reusable simulator

Implement simulate using this interface: `simulate(controller, target=lambda t: 40.0, load=lambda t: 0.0, duration=8.0, dt=0.001, noise=0.0, seed=42)`. Each log row is the state at the beginning of an interval and the command applied over that interval. Do not include a terminal-state row without a corresponding command.


**Required return fields:** `time`, `target`, `current`, `speed`, `measured`, `requested`, `voltage`, `p`, `i`, and `d`, each a one-dimensional array with one sample per interval. Here `i` is integral voltage, not winding current. With no saturation, requested and applied voltage are identical.

Call target and load with the interval start time. Add `noise * rng.normal()` only to the measured speed, using `numpy.random.default_rng(seed)`. Obtain P/I/D contributions from the controller after updating it. If it has a `requested` attribute, log that as requested voltage; otherwise use its return value. Reject a duration that is not an integer multiple of dt. Surface solver failures instead of using unsuccessful results.

</div>

<details>
<summary>Show a hint</summary>

At time k*dt, measure state, update once, log, then integrate to (k+1)*dt. Noise defaults to zero; its optional argument will be used later.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
def simulate(controller, target=lambda t: 40.0, load=lambda t: 0.0,
             duration=8.0, dt=0.001, noise=0.0, seed=42):
    # Each row describes the beginning of one control interval.
    if not np.isfinite(dt) or dt <= 0:
        raise ValueError("dt must be finite and positive")
    if not np.isfinite(duration) or duration <= 0:
        raise ValueError("duration must be finite and positive")
    if not np.isfinite(noise) or noise < 0:
        raise ValueError("noise must be finite and nonnegative")
    ratio = duration/dt
    if not np.isfinite(ratio):
        raise ValueError("duration/dt must be finite")
    count = round(ratio)
    if count < 1 or not np.isclose(ratio, count, rtol=0, atol=1e-9):
        raise ValueError("duration must be a positive integer multiple of dt")
    rng = np.random.default_rng(seed)
    state = np.array([0.0, 0.0])
    records = []
    for k in range(count):
        t = k*dt
        reference = target(t)
        measured = state[1] + noise*rng.normal()
        voltage = controller.update(reference, measured, dt)
        requested = getattr(controller, "requested", voltage)
        records.append([t, reference, state[0], state[1], measured,
                        requested, voltage, controller.p_term,
                        controller.i_term, controller.d_term])
        result = solve_ivp(motor_dynamics, (t, t+dt), state,
                          args=(voltage, load(t)), rtol=1e-7, atol=1e-9)
        if not result.success:
            raise RuntimeError(result.message)
        state = result.y[:, -1]
    keys = "time target current speed measured requested voltage p i d".split()
    return dict(zip(keys, np.asarray(records).T))

```

The final log is at duration−dt. Load and target schedules are sampled at interval starts; place their discontinuities on that grid. Measurements are separate from true speed so later noise experiments do not corrupt the physical state.

</details>

<div class="implementation-task" style="border-left:3px solid #b77979; background-color:rgba(183,121,121,0.08); padding:12px 18px; margin:16px 0;">

### Your implementation 5 — Plot a logged experiment

Implement plot_run(run,title) using three shared-time panels: target/true speed, requested/applied voltage, and P/I/D voltage contributions. Label axes and legends. This helper will be used in the next exercise.

</div>

<details>
<summary>Show a hint</summary>

The simulator returns a dictionary of equally sized arrays. Plot each requested field against time.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
def plot_run(run, title):
    fig, axes = plt.subplots(3, 1, sharex=True, figsize=(10, 8), layout="constrained")
    axes[0].plot(run['time'], run['target'], '--', label='Target')
    axes[0].plot(run['time'], run['speed'], label='True speed')
    axes[0].set_ylabel('Speed (rad/s)')
    axes[1].plot(run['time'], run['requested'], label='Requested')
    axes[1].plot(run['time'], run['voltage'], '--', label='Applied')
    axes[1].set_ylabel('Voltage (V)')
    for key in ['p', 'i', 'd']:
        axes[2].plot(run['time'], run[key], label=key.upper())
    axes[2].set_ylabel('Contribution (V)')
    axes[2].set_xlabel('Time (s)')
    for ax in axes:
        ax.grid(alpha=.25); ax.legend()
    fig.suptitle(title)
    plt.show()
```

Keeping plotting separate from simulation lets you inspect the same saved trajectory without rerunning the controller.

</details>

## 3.3 Predict proportional residual error

For a constant target and load, combine voltage balance, torque balance, and $V=K_p(r-\omega)$:

$$e_{ss}=\frac{(Rb+K_tK_e)r+R\tau_L}{Rb+K_tK_e+K_tK_p}.$$

The motor needs nonzero voltage to maintain speed against back EMF and friction. A pure P controller supplies that voltage only when error is nonzero. Increasing gain reduces the residual error but raises initial voltage demand.

### Where the persistent error comes from

Write $A=Rb+K_tK_e$ temporarily to shorten the algebra. The steady motor balance from Lesson 2 is $K_tV=A\omega+R\tau_L$. Substituting proportional control gives

$$K_tK_p(r-\omega)=A\omega+R\tau_L.$$

Collect speed terms and solve:

$$\omega_{ss}=\frac{K_tK_pr-R\tau_L}{A+K_tK_p}.$$

Subtract this expression from $r$ to obtain the error formula. With zero load, $r=40$, and $K_p=0.4$, the numerator of the error is $0.011(40)=0.44$ and the denominator is $0.011+0.04=0.051$. Error is therefore 8.627 rad/s and speed is 31.373 rad/s.

The corresponding command is $0.4(8.627)=3.451$ V. That is less than the 4.4 V needed to sustain the target speed. If error were exactly zero, P would request zero voltage; at positive speed, that cannot balance back EMF and friction. The residual is built into this controller, not a rounding error.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 6 — Check proportional residual error

Simulate gain 0.4 at 40 rad/s with no load. Plot the result and compare final error to the analytical prediction within 1%.

</div>

<details>
<summary>Show a hint</summary>

Use target minus the final true speed, not the measured voltage.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
run = simulate(PController(.4))
expected_error = (R*b+K_t*K_e)*40/(R*b+K_t*K_e+K_t*.4)
np.testing.assert_allclose(40-run['speed'][-1],expected_error,rtol=.01)
print('Predicted residual error:',expected_error)
plot_run(run,'P control: gain 0.4')
```

The residual is about 8.627 rad/s. This is expected model behavior, not a solver failure.

</details>

## 3.4 Compare gains fairly

Change one gain at a time while keeping the model, target, duration, and starting state identical. A fast response is not automatically a better controller: inspect command voltage as well as speed. High gains can excite oscillation and amplify measurement errors.

### Compare what the gain actually changes

Increasing gain from 0.2 to 0.8 reduces predicted residual error from about 14.194 to 4.835 rad/s. It also increases initial voltage demand from 8 to 32 V. The improvement is purchased with greater effort and a stronger reaction to measurement changes.

Inspect the early response at a zoomed time scale as well as the final value. Some gain changes affect the transient much more than the final plateau; others increase oscillation. A single endpoint cannot tell you the rise time, overshoot, or peak current.

The half-interval check is a numerical sensitivity check. When the sampled controller interval changes, the implemented system also changes slightly, so exact trajectory equality is not expected. Close final values for this experiment support using 1 ms here, not using it for every possible gain.

Save the distinction between **tracking accuracy**, **transient behavior**, and **effort**. The remaining lessons will improve different aspects, and those improvements can conflict.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 7 — Compare gains and sample intervals

Compare gains 0.2, 0.4, and 0.8. Print final error and peak voltage. Repeat gain 0.4 at dt=0.0005 and check final speed agrees within 0.01 rad/s.

</div>

<details>
<summary>Show a hint</summary>

Construct a fresh controller for every run; do not carry state across comparisons.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
for gain in [.2,.4,.8]:
    trial = simulate(PController(gain),duration=4)
    plt.plot(trial['time'],trial['speed'],label=f'Kp={gain}')
    print(gain,40-trial['speed'][-1],np.max(np.abs(trial['voltage'])))
plt.axhline(40,ls='--',color='gray');plt.xlabel('Time (s)');plt.ylabel('Speed (rad/s)');plt.legend();plt.show()
fine = simulate(PController(.4),dt=.0005)
assert abs(fine['speed'][-1]-run['speed'][-1]) < .01
```

The refinement checks one numerical comparison. It does not prove stability for arbitrary gains, delays, or motor parameters.

</details>

## What the experiment establishes

Negative feedback reduces the mismatch, but P control retains a predictable residual because the motor needs nonzero voltage at constant speed. The next lesson adds a memory of error so the controller can retain that voltage after present error disappears.

## Before moving on

Run your exercises from a fresh kernel in order. Check the stated numerical conditions and explain your plots in your own words: what changed, why it changed, and which model assumptions limit the conclusion. Reference solutions are one implementation, not the only correct approach.

[Previous lesson](02_open_loop_control.ipynb) · [Next lesson](04_integral_control.ipynb)

## References and further reading

[SciPy solve_ivp reference](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html) — callback arguments, output arrays, sample times, and tolerances.

[Åström and Murray, Feedback Systems, Chapter 11](https://www.cds.caltech.edu/~murray/books/AM08/pdf/fbs-pid_24Jul2020.pdf) — additional background on PID structure and implementation. Our examples use independent parallel gains and the conventions stated in this lesson.